# Safety-first triage acuity prediction

This notebook builds a simple triage-acuity model for the **Triagegeist** competition.

I treated the task less like a generic five-class classification problem and more like a safety audit. In triage, the error I care about most is **under-triage**: predicting that a patient is less urgent than they really are.

The goal here is a reproducible model that is easy to inspect:

- structured triage features;
- comorbidity history;
- chief complaint text;
- clear validation metrics;
- explicit under-triage and severe-case recall checks.

The data is synthetic, so I am careful not to make clinical-readiness claims. The model is a prototype and an audit exercise, not a deployable hospital system.


## Notebook map

1. Find and load the competition data.
2. Join patient history and chief complaint text.
3. Build a transparent baseline model.
4. Evaluate exact accuracy and safety-oriented errors.
5. Inspect which features push predictions toward each acuity class.
6. Generate a `submission.csv` artifact for reproducibility.
7. Add a short public-data context check using NHAMCS results from a separate local analysis.


## 1. Imports and data discovery

The notebook searches Kaggle's `/kaggle/input` mount first, then falls back to the local project folder. This keeps the same notebook runnable both on Kaggle and on my machine.


In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_fscore_support,
)
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

try:
    from IPython.display import display
except Exception:
    def display(obj):
        print(obj)

RANDOM_STATE = 42
TARGET = "triage_acuity"
ID_COL = "patient_id"
TEXT_COL = "chief_complaint_raw"
LEAKAGE_OR_UNAVAILABLE = {"disposition", "ed_los_hours", TARGET}


def looks_like_triagegeist_root(path: Path) -> bool:
    required = ["train.csv", "test.csv", "patient_history.csv", "chief_complaints.csv"]
    return path.exists() and all((path / name).exists() for name in required)


def find_data_root() -> Path:
    candidates = [
        Path("/kaggle/input/triagegeist"),
        Path("/kaggle/input") / "triagegeist",
        Path("data/raw/triagegeist"),
        Path("../data/raw/triagegeist"),
    ]
    for candidate in candidates:
        if looks_like_triagegeist_root(candidate):
            return candidate

    # Fallback: search under /kaggle/input in case Kaggle changes the mount name.
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        for candidate in kaggle_input.rglob("train.csv"):
            root = candidate.parent
            if looks_like_triagegeist_root(root):
                return root

    raise FileNotFoundError("Could not find Triagegeist data files.")

DATA_ROOT = find_data_root()
print(f"Using data root: {DATA_ROOT}")
print(sorted(p.name for p in DATA_ROOT.iterdir() if p.is_file()))


## 2. Load and join the data

The model uses only fields that are available at prediction time. I join the two auxiliary files by `patient_id`:

- `patient_history.csv`: comorbidity flags;
- `chief_complaints.csv`: raw chief complaint text.

I deliberately exclude `disposition` and `ed_los_hours`, because they are downstream outcomes and do not appear in the test file.


In [ ]:
def load_frame(data_root: Path, split: str) -> pd.DataFrame:
    frame = pd.read_csv(data_root / f"{split}.csv")
    history = pd.read_csv(data_root / "patient_history.csv")
    complaints = pd.read_csv(data_root / "chief_complaints.csv")

    # chief_complaint_system is already present in train/test; bring in raw text only.
    complaints = complaints[[ID_COL, TEXT_COL]]

    frame = frame.merge(history, on=ID_COL, how="left", validate="one_to_one")
    frame = frame.merge(complaints, on=ID_COL, how="left", validate="one_to_one")
    frame[TEXT_COL] = frame[TEXT_COL].fillna("")
    return frame

train = load_frame(DATA_ROOT, "train")
test = load_frame(DATA_ROOT, "test")

print(f"train shape after joins: {train.shape}")
print(f"test shape after joins:  {test.shape}")
display(train.head(3))


## 3. Quick data checks

The target is ESI-style acuity from 1 to 5, where 1 is most urgent and 5 is least urgent.


In [ ]:
target_distribution = (
    train[TARGET]
    .value_counts()
    .sort_index()
    .rename_axis("triage_acuity")
    .reset_index(name="count")
)
target_distribution["pct"] = target_distribution["count"] / len(train) * 100

display(target_distribution)

missing_summary = (
    train.drop(columns=[TARGET])
    .isna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
    .head(15)
    .rename("missing_pct")
    .reset_index()
    .rename(columns={"index": "column"})
)
display(missing_summary)


## 4. Feature lists and model pipeline

The feature set is intentionally simple:

- numeric triage and history fields;
- categorical intake/context fields;
- TF-IDF features from the chief complaint text.

The classifier is a balanced logistic regression model. I chose it because it is fast, stable, and inspectable. A more complex model might squeeze out extra performance, but that would not improve the clinical story as much as a clear error audit.


In [ ]:
def build_feature_lists(train_df: pd.DataFrame, test_df: pd.DataFrame) -> tuple[list[str], list[str], list[str]]:
    common = [c for c in train_df.columns if c in test_df.columns and c not in LEAKAGE_OR_UNAVAILABLE]
    common = [c for c in common if c != ID_COL]

    numeric_cols: list[str] = []
    categorical_cols: list[str] = []
    text_cols: list[str] = []

    for col in common:
        if col == TEXT_COL:
            text_cols.append(col)
        elif pd.api.types.is_numeric_dtype(train_df[col]):
            numeric_cols.append(col)
        else:
            categorical_cols.append(col)

    return numeric_cols, categorical_cols, text_cols


def make_pipeline(numeric_cols: list[str], categorical_cols: list[str], text_cols: list[str]) -> Pipeline:
    transformers: list[tuple[str, Any, Any]] = []

    transformers.append(
        (
            "numeric",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler(with_mean=False)),
                ]
            ),
            numeric_cols,
        )
    )

    transformers.append(
        (
            "categorical",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    (
                        "onehot",
                        OneHotEncoder(
                            handle_unknown="ignore",
                            min_frequency=10,
                            sparse_output=True,
                        ),
                    ),
                ]
            ),
            categorical_cols,
        )
    )

    if text_cols:
        transformers.append(
            (
                "chief_complaint_tfidf",
                TfidfVectorizer(
                    lowercase=True,
                    strip_accents="unicode",
                    ngram_range=(1, 2),
                    min_df=3,
                    max_df=0.98,
                    max_features=25_000,
                    sublinear_tf=True,
                ),
                text_cols[0],
            )
        )

    preprocessor = ColumnTransformer(transformers=transformers, remainder="drop", sparse_threshold=0.3)

    clf = LogisticRegression(
        C=1.5,
        class_weight="balanced",
        max_iter=1_000,
        random_state=RANDOM_STATE,
        solver="liblinear",
        penalty="l2",
    )

    return Pipeline(steps=[("features", preprocessor), ("model", clf)])

numeric_cols, categorical_cols, text_cols = build_feature_lists(train, test)
feature_cols = numeric_cols + categorical_cols + text_cols

print(f"Numeric features:     {len(numeric_cols)}")
print(f"Categorical features: {len(categorical_cols)}")
print(f"Text fields:          {len(text_cols)}")
print(f"Total source columns: {len(feature_cols)}")


## 5. Safety-oriented validation helpers

For ESI, lower numbers are more urgent. I count an error as **under-triage** when the predicted number is larger than the true number.

Example: true ESI 2 predicted as ESI 4 is under-triage.


In [ ]:
def clinical_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict[str, float]:
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    under = y_pred > y_true
    over = y_pred < y_true
    severe = y_true <= 2
    predicted_severe = y_pred <= 2

    return {
        "exact_match_rate": float((y_pred == y_true).mean()),
        "under_triage_rate": float(under.mean()),
        "over_triage_rate": float(over.mean()),
        "severe_esi_1_2_recall": float((y_pred[severe] <= 2).mean()) if severe.any() else np.nan,
        "severe_esi_1_2_precision": float((y_true[predicted_severe] <= 2).mean()) if predicted_severe.any() else np.nan,
        "severe_esi_1_2_under_triage_rate": float((y_pred[severe] > y_true[severe]).mean()) if severe.any() else np.nan,
        "mean_absolute_acuity_error": float(np.abs(y_pred - y_true).mean()),
        "within_one_level_rate": float((np.abs(y_pred - y_true) <= 1).mean()),
    }


def metric_table(metrics: dict[str, float]) -> pd.DataFrame:
    return pd.DataFrame(
        [{"metric": key, "value": value} for key, value in metrics.items()]
    )


## 6. Holdout validation

I use a stratified 80/20 validation split. This is not external validation, but it is enough to audit whether the pipeline is working and where its errors occur.


In [ ]:
X = train[feature_cols].copy()
y = train[TARGET].astype(int)

splitter = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
train_idx, valid_idx = next(splitter.split(X, y))

X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]

pipe = make_pipeline(numeric_cols, categorical_cols, text_cols)
pipe.fit(X_train, y_train)
valid_pred = pipe.predict(X_valid)

labels = [1, 2, 3, 4, 5]
summary_metrics = {
    "accuracy": accuracy_score(y_valid, valid_pred),
    "balanced_accuracy": balanced_accuracy_score(y_valid, valid_pred),
    "macro_f1": f1_score(y_valid, valid_pred, average="macro"),
    "weighted_f1": f1_score(y_valid, valid_pred, average="weighted"),
}
summary_metrics.update(clinical_metrics(y_valid.to_numpy(), valid_pred))

display(metric_table(summary_metrics))


## 7. Per-class performance and confusion matrix

The model performs extremely well on this synthetic dataset. I treat that as a reason to be cautious, not triumphant. Real ED data is usually much messier.


In [ ]:
precision, recall, f1, support = precision_recall_fscore_support(
    y_valid, valid_pred, labels=labels, zero_division=0
)
per_class = pd.DataFrame(
    {
        "triage_acuity": labels,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "support": support,
    }
)
display(per_class)

cm = confusion_matrix(y_valid, valid_pred, labels=labels)
cm_df = pd.DataFrame(cm, index=[f"actual_{x}" for x in labels], columns=[f"pred_{x}" for x in labels])
display(cm_df)

print(classification_report(y_valid, valid_pred, labels=labels, zero_division=0))


## 8. Coefficient audit

This is a linear model, so its coefficients are useful for a quick sanity check. I do **not** treat these as causal explanations. I use them to ask whether the model's strongest signals look clinically plausible.


In [ ]:
def coefficient_audit(fitted_pipe: Pipeline, top_n: int = 12) -> pd.DataFrame:
    feature_names = fitted_pipe.named_steps["features"].get_feature_names_out()
    model = fitted_pipe.named_steps["model"]
    rows = []

    for class_idx, class_label in enumerate(model.classes_):
        coefs = model.coef_[class_idx]
        top_positive = np.argsort(coefs)[-top_n:][::-1]
        for rank, idx in enumerate(top_positive, start=1):
            rows.append(
                {
                    "triage_acuity_class": int(class_label),
                    "rank": rank,
                    "feature": feature_names[idx],
                    "coefficient": float(coefs[idx]),
                }
            )

    return pd.DataFrame(rows)

coef_df = coefficient_audit(pipe, top_n=12)
display(coef_df)


A few examples from the coefficient table are reassuring:

- high-acuity classes are pushed by terms like shock, unresponsive, anaphylaxis, severe pain, and high NEWS2-style risk;
- low-acuity classes are pushed by administrative or mild complaint language such as advice, prescription, review, and referral.

That is exactly the kind of sanity check I would want before trusting any triage model enough to study further.


## 9. External public-data context: NHAMCS

Because Triagegeist is synthetic, I checked the feature design against a public real-world ED source: the **2022 National Hospital Ambulatory Medical Care Survey Emergency Department public-use file** from CDC/NCHS.

I did not use NHAMCS to train the Triagegeist model. I used it as a context check.

NHAMCS is not a perfect match. It uses coded reason-for-visit fields rather than raw chief complaint text, and its `IMMEDR` target is an urgency/immediacy analogue rather than the exact Triagegeist ESI target. But it contains the same broad triage ingredients: arrival context, vitals, pain, urgency/immediacy, reason for visit, and chronic conditions.

Local NHAMCS context check:

| Item | Value |
|---|---:|
| Public ED visit records | 16,025 |
| Columns | 913 |
| Survey-weighted estimated ED visits | about 155 million |

I also ran a separate NHAMCS-only binary model for immediate/emergent visits (`IMMEDR` 1-2) vs lower-acuity visits (`IMMEDR` 3-5). This was only supporting evidence, not direct validation.

| NHAMCS-only high-acuity metric | Value |
|---|---:|
| ROC AUC | 0.7757 |
| Average precision | 0.4593 |
| Baseline positive rate | 0.1701 |
| Recall at default threshold | 0.6429 |
| Balanced accuracy at default threshold | 0.7065 |
| Recall at high-recall threshold | 0.8018 |

The NHAMCS result is modest, which is what I would expect from real public ED data. I include it because it supports the basic premise: similar intake variables do carry urgency signal outside the synthetic competition data.


## 10. Fit final model and write outputs

There is no configured CSV leaderboard for this hackathon, but I still write a `submission.csv` so the notebook has a concrete, reproducible output.


In [ ]:
final_pipe = make_pipeline(numeric_cols, categorical_cols, text_cols)
final_pipe.fit(X, y)
test_pred = final_pipe.predict(test[feature_cols].copy()).astype(int)

submission = pd.DataFrame({ID_COL: test[ID_COL], TARGET: test_pred})
submission_path = Path("submission.csv")
submission.to_csv(submission_path, index=False)

metrics_payload = {
    "summary_metrics": {key: float(value) for key, value in summary_metrics.items()},
    "per_class": per_class.to_dict(orient="records"),
    "confusion_matrix": cm_df.to_dict(),
    "notes": "Synthetic Triagegeist validation only. NHAMCS used only as external public-data context.",
}
Path("validation_metrics.json").write_text(json.dumps(metrics_payload, indent=2))
coef_df.to_csv("coefficient_audit.csv", index=False)

display(submission.head())
print(f"Wrote {submission_path.resolve()}")
print(submission[TARGET].value_counts().sort_index())


## 11. Limitations

The biggest limitation is still the synthetic data. The validation performance is extremely high, and that probably means the simulated relationships are much cleaner than they would be in real ED operations.

I would not interpret this as a clinically ready model. I would interpret it as a reproducible prototype for how I would want to audit a triage model:

- track under-triage separately from over-triage;
- measure severe-case recall;
- inspect model signals;
- compare the feature design against public real-world ED data;
- avoid using downstream outcomes as predictors;
- keep the model advisory, not autonomous.

If this moved beyond a competition, the next step would be prospective validation on real local ED data with clinician review of every high-acuity miss.
